# Amazon Bedrock Runtime — Session Summary

## 1. Setting Up the Client

```python
bedrock = boto3.client("bedrock-runtime", region_name=region)
```

- This creates a local Python client object — **no network call is made**
- No AWS resource is started, opened, or charged
- It simply configures how to talk to the Bedrock Runtime API endpoint in `ap-southeast-2`
- Credentials from your IAM role are attached automatically (SigV4 signing)
- Think of it as saving a contact — not making a call

## 2. Two Bedrock Clients — Know the Difference

| Client | Purpose |
|---|---|
| `boto3.client("bedrock")` | **Management plane** — create/manage guardrails, models, inference profiles |
| `boto3.client("bedrock-runtime")` | **Data plane** — invoke models and run inference ✅ (what you use) |

## 3. Invoking the Embedding Model

```python
resp = bedrock.invoke_model(
    modelId="amazon.titan-embed-text-v2:0",
    body=json.dumps({"inputText": text, "normalize": True}),
    contentType="application/json",
    accept="application/json",
)
embeddings = json.loads(resp["body"].read())
```

- This is when AWS is **actually called** — an HTTPS POST to the Bedrock Runtime endpoint
- You are charged only here, based on input/output tokens
- The response body is a streaming object — `.read()` + `json.loads()` parses it into a Python dict

## 4. What Titan Embed Text v2 Does

- It is Amazon's own proprietary embedding model (not BERT)
- Takes your input text and returns a **1024-dimensional vector**
- That vector is a mathematical representation of the text's meaning

```
"customer orders"  →  [0.023, -0.081, 0.045, ...]  ← 1024 floats
```

## 5. Semantic Meaning in Vectors

| Text A | Text B | Vector Distance |
|---|---|---|
| "customer orders" | "purchase history" | Close (similar meaning) |
| "customer orders" | "cloud computing" | Far apart (different meaning) |

## 6. What `normalize=True` Does

- Scales the vector so its magnitude = 1 (unit vector)
- Makes cosine similarity comparisons faster and more accurate
- Best practice when storing vectors in a vector DB or doing semantic search

## 7. Billing

| Action | Charged? |
|---|---|
| `boto3.client("bedrock-runtime")` | ❌ No |
| `bedrock.invoke_model(...)` | ✅ Yes — per input/output tokens |
| Client sitting idle | ❌ No |

## 8. Typical Use Case (RAG Pipeline)

```
Input Text
    ↓
Titan Embed v2  (via bedrock.invoke_model)
    ↓
1024-dim vector
    ↓
Store in Vector DB (OpenSearch, pgvector, etc.)
    ↓
Semantic Search / Retrieval for your Chatbot
```

In [20]:
from dotenv import load_dotenv
import os
import boto3
import json
import sqlite3
load_dotenv()

True

In [21]:
embedding_model = os.getenv("EMBEDDING_MODEL_ID")
region = os.getenv("REGION")

bedrock = boto3.client("bedrock-runtime", region_name = region)

In [22]:
text = "customer orders"

resp = bedrock.invoke_model(
    modelId = embedding_model,
    body = json.dumps({"inputText": text, "normalize":True}),
    contentType="application/json",
    accept="application/json",
)

embeddings = json.loads(resp["body"].read())

In [24]:
embeddings['embedding'][:5]

[-0.09815262258052826,
 0.05554655194282532,
 0.01970723830163479,
 0.04128486290574074,
 -0.037085022777318954]

In [25]:
def embed(text: str) -> list:
    '''
    This function will basically convert the given text to a mathematical representation in the form of vector
    which will carry the semantics of the text
    This will help us to relate / find the closest match.
    '''

    response = bedrock.invoke_model(
        modelId = embedding_model,
        body = json.dumps({"inputText": text})
    )

    embeddings = json.loads(response['body'].read())['embedding']

    return embeddings

## Create the Vector Datata 

- Using ChromaDB

In [26]:
import chromadb
DESCRIPTIONS = {
    # existing tables
    "customers": "People who buy from the shop: name, city and signup date.",
    "products": "Items for sale with category and unit price.",
    "orders": "Customer purchases: which customer bought which product, quantity, date and status (completed, cancelled, returned).",
    # new tables
    "suppliers": "Companies that supply products to the shop, with country and quality rating.",
    "warehouses": "Storage locations where stock is kept.",
    "inventory": "Current stock quantity of each product in each warehouse.",
    "employees": "Shop staff with their role, department and hire date.",
    "departments": "Company departments such as sales, support and logistics.",
    "payments": "Payments made for orders: amount, payment method and payment status.",
    "shipments": "Delivery of orders: courier, shipped date, delivered date and delivery status.",
    "returns": "Products returned by customers with the return reason and refund amount.",
    "product_reviews": "Customer ratings and written reviews for products.",
    "promotions": "Discount campaigns with discount percentage and validity dates.",
    "stores": "Physical retail store locations with city and manager.",
    "invoices": "Billing documents issued for orders with tax and total amount.",
    "categories": "Product category hierarchy and category managers.",
    "loyalty_points": "Reward points earned and redeemed by customers.",
    "support_tickets": "Customer support requests with issue type, priority and open or closed status.",
}


In [27]:
con = sqlite3.connect(r'..\shop.db', uri=True)
tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
tables

['customers',
 'products',
 'orders',
 'suppliers',
 'warehouses',
 'inventory',
 'employees',
 'departments',
 'payments',
 'shipments',
 'returns',
 'product_reviews',
 'promotions',
 'stores',
 'invoices',
 'categories',
 'loyalty_points',
 'support_tickets']

In [28]:
con.execute('PRAGMA table_info(returns)').fetchall()

[(0, 'return_id', 'INTEGER', 0, None, 1),
 (1, 'order_id', 'INTEGER', 0, None, 0),
 (2, 'return_reason', 'TEXT', 0, None, 0),
 (3, 'refund_amount', 'REAL', 0, None, 0),
 (4, 'return_date', 'TEXT', 0, None, 0)]

In [29]:
client = chromadb.PersistentClient(path='chromadb')
try: 
    client.delete_collection('tables')
except:
    pass

col = client.create_collection('tables', metadata={'hsnw:space': 'cosine'})

for table in tables:
    columns = [c[1] for c in con.execute(f'PRAGMA table_info({table})').fetchall()]
    doc = f"Table {table}: {DESCRIPTIONS[table]} Columns: {", ".join(columns)}"
    col.add(ids=[table], documents=[doc], embeddings=[embed(doc)])
    print('indexed', table)

indexed customers
indexed products
indexed orders
indexed suppliers
indexed warehouses
indexed inventory
indexed employees
indexed departments
indexed payments
indexed shipments
indexed returns
indexed product_reviews
indexed promotions
indexed stores
indexed invoices
indexed categories
indexed loyalty_points
indexed support_tickets


In [32]:
col = client.get_collection('tables')
print(col.count())
print(col.metadata)

18
{'hsnw:space': 'cosine'}


In [36]:
row = col.get(ids=['products'], include = ['documents', 'embeddings', 'metadatas'])
row

{'ids': ['products'],
 'embeddings': array([[-0.10656527, -0.05281927,  0.05966846, ..., -0.03709202,
          0.00466616,  0.00572027]], shape=(1, 1024)),
 'documents': ['Table products: Items for sale with category and unit price. Columns: product_id, product_name, category, unit_price'],
 'uris': None,
 'included': ['documents', 'embeddings', 'metadatas'],
 'data': None,
 'metadatas': [None]}

## Retrieval Part